In [ ]:
# General
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import math

# Plotting
import matplotlib.pyplot as plt
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths

# SharePoint
path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Census Data')
path_main = os.path.join(path_sp, 'Data')

# Git
if user == 'jfontes':
    path_git     = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_config0 = os.path.join(path_git, 'config')
    path_config  = os.path.join(path_git, 'Python Code', 'Census', 'aa_config')
if user in ['jchoy', 'AAlAzzawi']:
    path_git     = os.path.join(path_users, 'Documents', 'Python Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_config0 = os.path.join(path_git, 'config')
    path_config  = os.path.join(path_git, 'Python Code', 'Census', 'aa_config')

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

In [ ]:
# Import objects
df_params = pd.read_excel(os.path.join(path_config, 'Census Configuration File.xlsx'), sheet_name = 'Inputs')

# Set parameters for querying Census data
indicator_name     = df_params[df_params['Type'] == 'indicator_name' ]['Input'].values[0]
estimate           = df_params[df_params['Type'] == 'estimate'       ]['Input'].values[0]
sample_type        = df_params[df_params['Type'] == 'sample'         ]['Input'].values[0]
geography          = df_params[df_params['Type'] == 'geography'      ]['Input'].values[0]
import_tab         = df_params[df_params['Type'] == 'import_tab'     ]['Input'].values[0]
percentages        = df_params[df_params['Type'] == 'percentages'    ]['Input'].values[0]
margin_of_error    = df_params[df_params['Type'] == 'margin_of_error']['Input'].values[0]
num_vars           = df_params[df_params['Type'] == 'num_vars'       ]['Input'].values[0]
year_start         = df_params[df_params['Type'] == 'year_start'     ]['Input'].values[0]
year_end           = df_params[df_params['Type'] == 'year_end'       ]['Input'].values[0]
report_theme       = df_params[df_params['Type'] == 'report_theme'   ]['Input'].values[0]
sp_folder_out      = df_params[df_params['Type'] == 'sp_folder'      ]['Input'].values[0]

# View
print(indicator_name)
print(estimate)
print(sample_type)
print(geography)
print(import_tab)
print("Percentages: " + percentages)
print("Margin of error: " + margin_of_error)
print("Number of variables: " + str(num_vars))
print(year_start)
print(year_end)

# Import about table
df_about = pd.read_excel(os.path.join(path_config0, 'About Indicators.xlsx'), sheet_name = 'Indicators')
df_about = df_about[df_about['Indicator'] == indicator_name]
folder = df_about.Folder.values[0]
MOE_thresh = df_about['MOE Threshold'].values[0]
print(folder)
print('MOE threshold: ' + str(MOE_thresh) + '%')

## Import Data

In [ ]:
# Import data by geography
path_in = os.path.join(path_main, report_theme, sp_folder_out, indicator_name + ' ' + folder)

try: 
    df_counties = pd.read_excel(os.path.join(path_in, indicator_name + ' Counties ' + estimate + '.xlsx'), sheet_name = 'Counties')
    display(df_counties.head(3))
except Exception as e: print(e)

try: 
    df_mpo = pd.read_excel(os.path.join(path_in, indicator_name + ' MPO ' + estimate + '.xlsx'), sheet_name = 'MPO')
    display(df_mpo.head(3))
except Exception as e: print(e)

try: 
    df_msa = pd.read_excel(os.path.join(path_in, indicator_name + ' MSA ' + estimate + '.xlsx'), sheet_name = 'MSA')
    display(df_msa.head(3))
except Exception as e: print(e)

try: 
    df_counties = pd.read_excel(os.path.join(path_in, indicator_name + ' Counties ' +  re.sub('ACS', 'PUMS', estimate) + '.xlsx'), sheet_name = 'Counties')
    display(df_counties.head(3))
except Exception as e: print(e)

try: 
    df_mpo = pd.read_excel(os.path.join(path_in, indicator_name + ' MPO ' +  re.sub('ACS', 'PUMS', estimate) + '.xlsx'), sheet_name = 'MPO')
    display(df_mpo.head(3))
except Exception as e: print(e)

In [ ]:
if indicator_name == 'Cost_3':
    if geography == 'Tracts':
        df_tracts = df_tracts[df_tracts['Variable'] == 'Vacant']
    if geography == 'Counties':
        df_counties = df_counties[df_counties['Variable'] == 'Vacant']
        df_mpo      = df_mpo     [df_mpo     ['Variable'] == 'Vacant']
    if geography == 'MSA':
        df_msa = df_msa[df_msa['Variable'] == 'Vacant']

if indicator_name == 'Cost_5':
    if geography == 'Tracts':
        df_tracts = df_tracts[df_tracts['Variable'] == 'Owner occupied']
    if geography == 'Counties':
        df_counties = df_counties[df_counties['Variable'] == 'Owner occupied']
        df_mpo      = df_mpo     [df_mpo     ['Variable'] == 'Owner occupied']
    if geography == 'MSA':
        df_msa = df_msa[df_msa['Variable'] == 'Owner occupied']

if indicator_name == 'Health_2':
    if geography == 'Tracts':
        df_tracts = df_tracts[df_tracts['Variable'] == 'No health insurance coverage']
    if geography == 'Counties':
        df_counties = df_counties[df_counties['Variable'] == 'No health insurance coverage']
        df_mpo      = df_mpo     [df_mpo     ['Variable'] == 'No health insurance coverage']
    if geography == 'MSA':
        df_msa = df_msa[df_msa['Variable'] == 'No health insurance coverage']

if indicator_name == 'Income_4':
    if geography == 'Tracts':
        df_tracts = df_tracts[df_tracts['Variable'] == 'Total Income in the past 12 months below poverty level']
    if geography == 'Counties':
        df_counties = df_counties[df_counties['Variable'] == 'Total Income in the past 12 months below poverty level']
        df_mpo      = df_mpo     [df_mpo     ['Variable'] == 'Total Income in the past 12 months below poverty level']
    if geography == 'MSA':
        df_msa = df_msa[df_msa['Variable'] == 'Total Income in the past 12 months below poverty level']

if indicator_name == 'Cost_6':
    # df_counties = df_counties[df_counties['housing_type'] == 'Owners and Renters']
    # df_mpo      = df_mpo     [df_mpo     ['housing_type'] == 'Owners and Renters']
    df_mpo = df_mpo[(df_mpo['housing_type'] == 'Owner') | (df_mpo['housing_type'] == 'Renter')]
    df_counties = df_counties[(df_counties['housing_type'] == 'Owner') | (df_counties['housing_type'] == 'Renter')]

if indicator_name == 'Accessibility_4':
    df_mpo = df_mpo[
                    (df_mpo['Income Bracket'].isin(['Low Income', 'Moderate Income', 'High Income']))
                    & (df_mpo['RAC1P'] == 'All')
                     ]
    


In [ ]:
# remove state FIPS field
# remove "All" category for race/ethnicity
# make sure index is removed
# make sure proportions are now percentages

try:
    df_counties = df_counties.drop(['State FIPS', 'County FIPS'], axis = 1)
    df_mpo      = df_mpo     .drop(['State FIPS'               ], axis = 1)
    
    df_counties.columns = [col.lower() for col in df_counties.columns]
    df_mpo     .columns = [col.lower() for col in df_mpo     .columns]
    
    df_counties.columns = [re.sub('[\s+]', '_', col.strip()) for col in df_counties.columns]
    df_mpo     .columns = [re.sub('[\s+]', '_', col.strip()) for col in df_mpo     .columns]
    
    df_counties.columns = [re.sub('\\?'  , '' , col.strip()) for col in df_counties.columns]
    df_mpo     .columns = [re.sub('\\?'  , '' , col.strip()) for col in df_mpo     .columns]
except:
    pass

try:      
    df_msa  = df_msa.reset_index(drop = True)
    df_msa.columns = [x.lower() for x in df_msa.columns]
    df_msa.columns = [re.sub('[\s+]', '_', col.strip()) for col in df_msa.columns]
    df_msa.columns = [re.sub('\\?'  , '' , col.strip()) for col in df_msa.columns]
except:
    pass
    
if indicator_name == 'Cost_6':
    df_counties.loc[df_counties['housing_burden'] == 'Cost burden <=30%'        , 'housing_burden'] = 'Cost burden less than 30 perc'
    df_counties.loc[df_counties['housing_burden'] == 'Cost burden >30% to <=50%', 'housing_burden'] = 'Cost burden 30 to 50 perc'
    df_counties.loc[df_counties['housing_burden'] == 'Cost burden >50%'         , 'housing_burden'] = 'Cost burden greater than 50 perc'
    
    df_mpo.loc[df_mpo['housing_burden'] == 'Cost burden <=30%'        , 'housing_burden'] = 'Cost burden less than 30 perc'
    df_mpo.loc[df_mpo['housing_burden'] == 'Cost burden >30% to <=50%', 'housing_burden'] = 'Cost burden 30 to 50 perc'
    df_mpo.loc[df_mpo['housing_burden'] == 'Cost burden >50%'         , 'housing_burden'] = 'Cost burden greater than 50 perc'

    

In [ ]:
try:
    print('Columns: ' + str(list(df_mpo.columns)))
    print('Variables: '      + str(unique(df_mpo.variable      .values)))
    print('Race/Ethnicity: ' + str(unique(df_mpo.race_ethnicity.values)))
    display(df_counties.head(3), df_mpo.head(3))
except:
    pass

try:
    print('Columns: ' + str(list(df_msa.columns)))
    print('Variables: '      + str(unique(df_msa.variable      .values)))
    print('Race/Ethnicity: ' + str(unique(df_msa.race_ethnicity.values)))
    display(df_msa.head(3))
except:
    pass

try:
    print('Columns: ' + str(list(df_mpo.columns)))
    print('Variables: '      + str(unique(df_mpo.income_bracket.values)))
    display(df_counties.head(3), df_mpo.head(3))
except:
    pass

## Data Visualization

#### Line Graphs

In [ ]:
path_plots = os.path.join(path_main, report_theme, sp_folder_out, indicator_name + ' ' + folder, 'plots')
# df_plot = df_counties.copy()
df_plot = df_mpo.copy()
# df_plot = df_msa.copy()


# Plotting setup
by_race = False
race_ethnicity = 'rac1p'
loop_vars = False
variable = 'variable'
x = 'year'
y = 'percentage'
color = 'income_bracket'
line_dash = 'travel_time'
markers = True
plot_title = folder + ' ' + 'SACOG MPO' + ' ' + estimate
plot_name  = folder + '_' + 'SACOG MPO' + '_' + estimate
# plot_title = 'Percent of regional median household income' + estimate + ' SACOG MPO'
# plot_name  = 'Percent of regional median household income' + estimate + ' SACOG MPO'
export = True

In [ ]:
def plot_lines(
    df=df_plot
     , loop_vars=loop_vars, by_race=by_race, race_ethnicity=race_ethnicity, variable=variable
     , x=x, y=y
     , color=color, line_dash=line_dash, markers=markers
     , plot_title=plot_title, plot_name=plot_name
     , export=export
):
    if race_ethnicity != None:
        if by_race:
            df = df[df[race_ethnicity] != 'All']
        else:
            df = df[df[race_ethnicity] == 'All'].drop(race_ethnicity, axis = 1)  

    if loop_vars:
        vars = unique(df[variable].values)
        for var in vars:
            df2 = df[df[variable] == var]
            fig = px.line(df2, x = x, y = y, color = color, line_dash = line_dash, markers = markers)
            fig.update_layout(title = plot_title + ' - ' + str(var))
            if export:
                fig.write_html(os.path.join(path_plots, ''.join([indicator_name + '_', plot_name + '_', var + '_', 'line.html'])))
                
            fig.update_layout(autosize=False, width=1050, height=450)
            
    else:
        fig = px.line(df, x = x, y = y, color = color, line_dash = line_dash, markers = markers)
        fig.update_layout(title = plot_title)
        if export:
            fig.write_html(os.path.join(path_plots, ''.join([indicator_name + '_', plot_name + '_', 'line.html'])))

        fig.update_layout(autosize=False, width=1050, height=450)

    return fig.show()
        
plot_lines()

***

## Accessibility_4

***

In [ ]:
indicator_name = 'Accessibility_4'
estimate = 'ACS1'
# estimate = 'ACS5'

In [ ]:
df_acs_p = pd.read_csv(os.path.join(path_agol, indicator_name, 'Inter', indicator_name + '_PUMA_' + estimate + '_P.csv')
                                    , dtype = {'State FIPS': object, 'PUMA': object, 'SERIALNO': object})
df_acs_p = df_acs_p[df_acs_p['SPORDER'] == 1]
df_acs_p = df_acs_p[df_acs_p['JWTRNS'] != 'N/A, not a civillian in the labor force']

df_acs_h = pd.read_csv(os.path.join(path_agol, indicator_name, 'Inter', indicator_name + '_PUMA_' + estimate + '_H.csv')
                                    , dtype = {'State FIPS': object, 'PUMA': object, 'SERIALNO': object})
df_acs_h = df_acs_h[(df_acs_h['NP'] > 0) & (df_acs_h['WGTP'] > 0)]


df_cpi = pd.read_excel(os.path.join(path_config0, 'CPI Inflation Adjustment Factors.xlsx'), sheet_name = 'BLS_West')
df_cpi = df_cpi[['Year', 'IAF_2023']].rename(columns = {'Year':'year'})

In [ ]:
df_acs_h = df_acs_h.merge(df_acs_p.drop(['SPORDER', 'PWGTP'], axis = 1), on = ['State FIPS', 'PUMA', 'PUMA NAME', 'SERIALNO', 'year'], how = 'left')
df_acs_h = df_acs_h.dropna(subset = ['JWTRNS'])

df_acs_h = df_acs_h.merge(df_cpi, on = 'year', how = 'left')
df_acs_h['HINCP'] = df_acs_h['HINCP']*df_acs_h['ADJINC']*df_acs_h['IAF_2023']
df_acs_h = df_acs_h.drop(['IAF_2023', 'ADJINC'], axis = 1)

df_acs = df_acs_h.copy()

df_fips1 = pd.read_excel(os.path.join(path_config0, 'Area Codes.xlsx'), sheet_name = 'CountyFIPS' 
                                , dtype = {'State FIPS': object, 'County FIPS': object})
df_fips2 = pd.read_excel(os.path.join(path_config0, 'Area Codes.xlsx'), sheet_name = 'PUMAcodes' 
                                , dtype = {'STATEFP': object, 'COUNTYFP': object})

df_fips1 = df_fips1[df_fips1['State FIPS'].isin(['06'])]
df_fips2 = df_fips2[df_fips2['STATEFP'   ].isin(['06'])]

df_fips2['PUMA5CE'] = df_fips2['PUMA5CE'].astype(str).apply('{:0>5}'.format)
df_fips2 = df_fips2[['STATEFP', 'COUNTYFP', 'PUMA5CE', 'Years']].drop_duplicates()
df_fips2 = df_fips2.rename(columns = {'STATEFP':'State FIPS', 'COUNTYFP':'County FIPS', 'PUMA5CE':'PUMA'})

df_acs['PUMA'] = df_acs['PUMA'].astype(str).apply('{:0>5}'.format)

df_fips1 = df_fips1[['State FIPS', 'County FIPS', 'County Name', 'MPO']]
# df_fips1 = df_fips1[['State FIPS', 'County FIPS', 'County Name', 'MSA']]

df_acs2 = df_acs[df_acs['year'].isin(sequence(2022, 2031, 1))].merge(df_fips2[df_fips2['Years'] == '2022-2031'], on = ['State FIPS', 'PUMA'])
df_acs1 = df_acs[df_acs['year'].isin(sequence(2012, 2021, 1))].merge(df_fips2[df_fips2['Years'] == '2012-2021'], on = ['State FIPS', 'PUMA'])
df_acs = pd.concat([df_acs1, df_acs2])

df_acs = df_acs.merge(df_fips1, on = ['State FIPS', 'County FIPS'])
df_acs = df_acs.drop(['SERIALNO', 'Years'], axis = 1)

df_acs.head()

In [ ]:
df_income_brackets = pd.read_excel(os.path.join(path_config0, 'CA State Income Brackets by Household Size.xlsx'), sheet_name = 'Table')
df_income_brackets['County'].fillna(method='ffill', inplace = True)
df_income_brackets['County'] = df_income_brackets['County'].str.replace(' County.*'         , '' , regex = True)
df_income_brackets['County'] = df_income_brackets['County'].str.replace('\n'                , ' ', regex = True)
df_income_brackets['AMI'   ] = df_income_brackets['County'].str.extract('\$?([0-9,]+)[.%]?')
df_income_brackets['AMI'   ] = df_income_brackets['AMI'   ].str.replace(','                 , '' , regex = True)
df_income_brackets['County'] = df_income_brackets['County'].str.replace(' \$?([0-9,]+)[.%]?', '' , regex = True)
df_income_brackets = pd.melt(df_income_brackets
                              , id_vars = ['County', 'Income Bracket', 'AMI']
                              , var_name = 'NP'
                              , value_name = 'Income Threshold'
                            )
df_income_brackets = df_income_brackets[df_income_brackets['Income Bracket'].isin(['Low Income', 'Moderate Income'])]
df_income_brackets = df_income_brackets.pivot_table(index = ['County', 'NP']
                                       , columns = 'Income Bracket'
                                       , values = 'Income Threshold').reset_index().rename(columns = {'County':'County Name'})

df_acs_jwmnp = df_acs.merge(df_income_brackets, on = ['County Name', 'NP'], how = 'left')


df_acs_jwmnp.loc[ df_acs_jwmnp['HINCP'] <= df_acs_jwmnp['Low Income']                                                              , 'Income Bracket'] = 'Low Income'
df_acs_jwmnp.loc[(df_acs_jwmnp['HINCP']  > df_acs_jwmnp['Low Income']) & (df_acs_jwmnp['HINCP'] <= df_acs_jwmnp['Moderate Income']), 'Income Bracket'] = 'Moderate Income'
df_acs_jwmnp.loc[ df_acs_jwmnp['HINCP']  > df_acs_jwmnp['Moderate Income']                                                         , 'Income Bracket'] = 'High Income'
df_acs_jwmnp.loc[ df_acs_jwmnp['NP'] == 0                                                                                          , 'Income Bracket'] = 'No data available'

df_acs_jwmnp.head()

In [ ]:
# df_rep = df_acs_jwmnp[['State FIPS', 'MPO', 'County Name', 'year', 'RAC1P', 'Income Bracket', 'WGTP', 'JWMNP']]
# df_rep = df_rep.groupby(['State FIPS', 'MPO', 'year','Income Bracket', 'JWMNP'], as_index = False)['WGTP'].agg(sum)
df_rep = df_acs[['State FIPS', 'MPO', 'County Name', 'year', 'RAC1P', 'WGTP', 'HINCP']]
df_rep = df_rep.groupby(['State FIPS', 'MPO', 'year', 'HINCP'], as_index = False)['WGTP'].agg(sum)
df_rep.head()

In [ ]:
list_keys = []
for list_ in df_rep.values:
    list_keys.append(tuple(list_[:-1]))

list_values = []
for list_ in df_rep.values:
    list_values.append(list_[-1])

dict_replicates = dict(zip(list_keys, list_values))

list_df = []
for tuple_ in tqdm(list(dict_replicates.keys())):
    
    row  = list(tuple_)
    wgtp = dict_replicates[tuple_]

    list_df.append(pd.concat([pd.DataFrame(row).T] * wgtp))

df_rep = pd.concat(list_df)
# df_rep.columns = ['State FIPS', 'MPO',  'year', 'Income Bracket', 'JWMNP']
df_rep.columns = ['State FIPS', 'MPO',  'year', 'HINCP']
df_rep.head()

In [ ]:
# if estimate == 'ACS5':
#     df_plot = df_rep[df_rep['year'].isin([2019, 2020, 2021])]
# if estimate == 'ACS1':
#     df_plot = df_rep[df_rep['year'].isin([2019, 2021, 2022])]
# df_plot = df_plot[df_plot['Income Bracket'] != 'Moderate Income']
# fig = px.histogram(df_plot, x="JWMNP", facet_col='year', facet_row='Income Bracket', nbins=40)
# fig['layout']['xaxis' ]['title']['text']=''
# fig['layout']['xaxis2']['title']['text']='Commute Times (Minutes)'
# fig['layout']['xaxis3']['title']['text']=''
# fig.show()


if estimate == 'ACS5':
    df_plot = df_rep[df_rep['year'].isin([2019, 2020, 2021])]
if estimate == 'ACS1':
    df_plot = df_rep[df_rep['year'].isin([2019, 2021, 2022])]
fig = px.histogram(df_plot, x="HINCP", facet_col='year', nbins=120)
fig['layout']['xaxis' ]['title']['text']=''
fig['layout']['xaxis2']['title']['text']='Household Income (2023 Inflation Adjustied Dollars ($))'
fig['layout']['xaxis3']['title']['text']=''
fig.show()

In [ ]:
path_plots = os.path.join(path_main, report_theme, sp_folder_out, indicator_name + ' ' + folder, 'plots')
# fig.write_html(os.path.join(path_plots, '_'.join([indicator_name, 'Commute Times by Year', 'SACOG MPO', re.sub('ACS', 'PUMS', estimate), 'histogram.html'])))
fig.write_html(os.path.join(path_plots, '_'.join(['Income_1', 'Household Income by Year', 'SACOG MPO', re.sub('ACS', 'PUMS', estimate), 'histogram.html'])))